# Semantic Trend Embedding + XGBoost for CMAPSS Near-Failure Detection

This notebook is the article-ready version of the earlier `llm_cmapss_anomaly_detection.ipynb` idea.

Key corrections for publication:

1. The task is treated as **supervised near-failure classification**, not unsupervised anomaly detection.
2. Train/validation splitting is performed at engine level before fitting learned preprocessing.
3. FD002/FD004 operating-condition normalization is fit only on the training engines, then applied to validation/test.
4. The semantic contribution is tested with ablations: numeric-only, semantic-only, and hybrid.
5. FD002/FD004 condition clusters are used as one-hot model features and as context in semantic trend text.
6. Numeric features use multi-window statistics at 10, 30, and 60 cycles.
7. Probability thresholds are selected per condition cluster on the validation split.
8. Tables, figures, and a short article draft are exported to `article_assets/semantic_trend_xgboost/`.


## 1. Configuration

If a package is missing, run the optional install cell first. For the paper, report the exact versions printed by this notebook.

In [ ]:
# Optional dependency install. Run only if your environment is missing packages.
# %pip install -q numpy pandas scikit-learn xgboost sentence-transformers matplotlib


In [ ]:
# Google Colab Drive mount. Outside Colab, this cell is skipped safely.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; Drive mount skipped.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import math
import os
import platform
import random
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print(f"Matplotlib is unavailable; figures will be skipped: {exc}")

SEED = 42
BASE = "/content/drive/MyDrive/data"
DATA_DIR = Path(BASE)
OUT_DIR = Path("article_assets") / "semantic_trend_xgboost"
DATASETS = ["FD001", "FD002", "FD003", "FD004"]
METHODS = ["numeric", "semantic", "hybrid"]

RUL_THRESHOLD = 30
WINDOW_SIZES = [10, 30, 60]
TEXT_WINDOW_SIZE = 30
CONDITION_AWARE_THRESHOLDS = True
VAL_ENGINE_RATIO = 0.20
TOP_K_TEXT_SENSORS = 8
N_CONDITIONS = 6
MIN_SENSOR_VAR = 1e-6
N_ESTIMATORS = 800
EMBED_BATCH_SIZE = 256
HF_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MULTI_CONDITION_DATASETS = {"FD002", "FD004"}

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Data dir:", DATA_DIR.resolve())
print("Output dir:", OUT_DIR.resolve())


Python: 3.12.13
Data dir: /content/drive/MyDrive/data
Output dir: /content/article_assets/semantic_trend_xgboost


## 2. Leakage-Controlled Data Preparation

In [ ]:
def load_cmapss(dataset: str, data_dir: Path):
    train_df = pd.read_csv(data_dir / f"train_{dataset}.csv").dropna(axis=1, how="all")
    test_df = pd.read_csv(data_dir / f"test_{dataset}.csv").dropna(axis=1, how="all")
    rul_df = pd.read_csv(data_dir / f"RUL_{dataset}.txt", header=None, names=["final_rul"])

    max_cycle = train_df.groupby("unit_number")["time_in_cycles"].max().rename("max_cycle")
    train_df = train_df.merge(max_cycle, on="unit_number")
    train_df["RUL"] = train_df["max_cycle"] - train_df["time_in_cycles"]
    train_df.drop(columns=["max_cycle"], inplace=True)

    unit_order = np.sort(test_df["unit_number"].unique())
    if len(unit_order) != len(rul_df):
        raise ValueError(f"{dataset}: test engine count does not match RUL file")
    final_rul_map = dict(zip(unit_order, rul_df["final_rul"].astype(float)))
    test_df["final_rul"] = test_df["unit_number"].map(final_rul_map)
    max_cycle_test = test_df.groupby("unit_number")["time_in_cycles"].transform("max")
    test_df["RUL"] = test_df["final_rul"] + (max_cycle_test - test_df["time_in_cycles"])

    op_cols = [c for c in train_df.columns if c.startswith("operational_setting_")]
    sensor_cols = [c for c in train_df.columns if c.startswith("sensor_measurement_")]
    return train_df, test_df, op_cols, sensor_cols


def split_train_validation(train_df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    units = np.sort(train_df["unit_number"].unique())
    train_units, val_units = train_test_split(
        units,
        test_size=VAL_ENGINE_RATIO,
        random_state=SEED,
        shuffle=True,
    )
    return np.sort(train_units), np.sort(val_units)


def select_sensor_columns(train_part: pd.DataFrame, sensor_cols: Sequence[str]) -> List[str]:
    variances = train_part[list(sensor_cols)].var(numeric_only=True)
    return [c for c in sensor_cols if pd.notna(variances[c]) and variances[c] > MIN_SENSOR_VAR]


def fit_condition_normalizer(train_part, op_cols, sensor_cols):
    km = KMeans(n_clusters=N_CONDITIONS, random_state=SEED, n_init=10)
    labels = km.fit_predict(train_part[list(op_cols)].values)
    scalers: Dict[int, StandardScaler] = {}
    for cluster_id in range(N_CONDITIONS):
        mask = labels == cluster_id
        scaler = StandardScaler()
        fit_rows = train_part.loc[mask, list(sensor_cols)] if np.any(mask) else train_part[list(sensor_cols)]
        scaler.fit(fit_rows.fillna(0.0))
        scalers[cluster_id] = scaler
    return km, scalers


def apply_condition_normalizer(df, op_cols, sensor_cols, km, scalers):
    out = df.copy()
    labels = km.predict(out[list(op_cols)].values)
    out["_condition_cluster"] = labels.astype(int)
    for cluster_id, scaler in scalers.items():
        mask = labels == cluster_id
        if np.any(mask):
            out.loc[mask, list(sensor_cols)] = scaler.transform(out.loc[mask, list(sensor_cols)].fillna(0.0))
    return out


def add_single_condition_cluster(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["_condition_cluster"] = 0
    return out


def condition_one_hot(condition_labels: Sequence[int]) -> pd.DataFrame:
    labels = np.asarray(condition_labels, dtype=int)
    data = np.zeros((len(labels), N_CONDITIONS), dtype=np.float32)
    valid = (labels >= 0) & (labels < N_CONDITIONS)
    data[np.arange(len(labels))[valid], labels[valid]] = 1.0
    columns = [f"condition_cluster_{i}" for i in range(N_CONDITIONS)]
    return pd.DataFrame(data, columns=columns)


def maybe_normalize_conditions(dataset, train_part, val_part, test_df, op_cols, sensor_cols):
    if dataset not in MULTI_CONDITION_DATASETS:
        return add_single_condition_cluster(train_part), add_single_condition_cluster(val_part), add_single_condition_cluster(test_df)
    km, scalers = fit_condition_normalizer(train_part, op_cols, sensor_cols)
    return (
        apply_condition_normalizer(train_part, op_cols, sensor_cols, km, scalers),
        apply_condition_normalizer(val_part, op_cols, sensor_cols, km, scalers),
        apply_condition_normalizer(test_df, op_cols, sensor_cols, km, scalers),
    )


## 3. Window Statistics and Semantic Trend Texts

In [ ]:
def rolling_slope(values: np.ndarray, window_size: int) -> np.ndarray:
    values = values.astype(np.float64)
    n = len(values)
    if n == 0:
        return np.array([], dtype=np.float64)
    x = np.arange(n, dtype=np.float64)
    pad = np.array([0.0], dtype=np.float64)
    cy = np.concatenate([pad, np.cumsum(values)])
    cx = np.concatenate([pad, np.cumsum(x)])
    cx2 = np.concatenate([pad, np.cumsum(x * x)])
    cxy = np.concatenate([pad, np.cumsum(x * values)])

    ends = np.arange(1, n + 1)
    starts = np.maximum(0, ends - window_size)
    count = (ends - starts).astype(np.float64)
    sum_y = cy[ends] - cy[starts]
    sum_x = cx[ends] - cx[starts]
    sum_x2 = cx2[ends] - cx2[starts]
    sum_xy = cxy[ends] - cxy[starts]
    denom = count * sum_x2 - sum_x * sum_x
    numer = count * sum_xy - sum_x * sum_y
    return np.divide(numer, denom, out=np.zeros_like(numer), where=denom > 0.0)


def extract_window_features(df: pd.DataFrame, sensor_cols: Sequence[str], window_size: int) -> pd.DataFrame:
    columns: List[str] = []
    for col in sensor_cols:
        columns.extend([
            f"{col}_w{window_size}_mean",
            f"{col}_w{window_size}_std",
            f"{col}_w{window_size}_slope",
            f"{col}_w{window_size}_max",
            f"{col}_w{window_size}_min",
        ])

    parts = []
    sorted_df = df.sort_values(["unit_number", "time_in_cycles"])
    for _, group in sorted_df.groupby("unit_number", sort=False):
        feature_values = np.zeros((len(group), len(columns)), dtype=np.float32)
        offset = 0
        for col in sensor_cols:
            series = pd.Series(group[col].astype(float).values)
            feature_values[:, offset + 0] = series.rolling(window_size, min_periods=1).mean().values
            feature_values[:, offset + 1] = series.rolling(window_size, min_periods=1).std(ddof=0).fillna(0.0).values
            feature_values[:, offset + 2] = rolling_slope(series.values, window_size)
            feature_values[:, offset + 3] = series.rolling(window_size, min_periods=1).max().values
            feature_values[:, offset + 4] = series.rolling(window_size, min_periods=1).min().values
            offset += 5
        parts.append(pd.DataFrame(feature_values, index=group.index, columns=columns))
    return pd.concat(parts).loc[df.index]


def extract_multi_window_features(df: pd.DataFrame, sensor_cols: Sequence[str]) -> pd.DataFrame:
    return pd.concat(
        [extract_window_features(df, sensor_cols, window_size) for window_size in WINDOW_SIZES],
        axis=1,
    )


def slope_phrase(slope: float, std: float, sensor_name: str) -> str:
    abs_slope = abs(float(slope))
    direction = "increasing" if slope > 0 else "decreasing"
    if abs_slope > 0.10:
        speed = "rapidly"
    elif abs_slope > 0.02:
        speed = "steadily"
    elif abs_slope > 0.005:
        speed = "slowly"
    else:
        return f"{sensor_name} stable std {std:.3f}"
    return f"{sensor_name} {speed} {direction} slope {slope:+.3f} std {std:.3f}"


def build_texts(win_df: pd.DataFrame, sensor_cols: Sequence[str], condition_labels: Sequence[int]) -> List[str]:
    slope_cols = [f"{c}_w{TEXT_WINDOW_SIZE}_slope" for c in sensor_cols]
    std_cols = [f"{c}_w{TEXT_WINDOW_SIZE}_std" for c in sensor_cols]
    slopes = win_df[slope_cols].values
    stds = win_df[std_cols].values
    sensor_labels = [c.replace("sensor_measurement_", "sensor_") for c in sensor_cols]
    condition_labels = np.asarray(condition_labels, dtype=int)
    k = min(TOP_K_TEXT_SENSORS, len(sensor_cols))

    texts = []
    for row_idx in range(slopes.shape[0]):
        chosen = np.argsort(np.abs(slopes[row_idx]))[-k:][::-1] if k < len(sensor_cols) else np.arange(len(sensor_cols))
        phrases = [slope_phrase(slopes[row_idx, j], stds[row_idx, j], sensor_labels[j]) for j in chosen]
        texts.append(
            f"Operating condition cluster {condition_labels[row_idx]}. "
            + "Engine degradation trend: "
            + "; ".join(phrases)
            + "."
        )
    return texts


## 4. Model Training, Threshold Selection, and Metrics

In [ ]:
class TextEmbedder:
    def __init__(self, model_name: str, cache_dir: Path):
        from sentence_transformers import SentenceTransformer
        self.model_name = model_name
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.model = SentenceTransformer(model_name)

    def encode(self, texts: Sequence[str], cache_key: str) -> np.ndarray:
        cache_file = self.cache_dir / f"{cache_key}.npy"
        if cache_file.exists():
            return np.load(cache_file)
        emb = self.model.encode(
            list(texts),
            batch_size=EMBED_BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype(np.float32)
        np.save(cache_file, emb)
        return emb


def labels_from_rul(rul: np.ndarray) -> np.ndarray:
    return (rul <= RUL_THRESHOLD).astype(int)


def best_threshold_f1(y_true: np.ndarray, y_score: np.ndarray) -> Tuple[float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f1 = (2.0 * precision * recall) / (precision + recall + 1e-12)
    idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[idx]), float(f1[idx])


def metric_row(y_true: np.ndarray, y_score: np.ndarray, threshold: float, prefix: str) -> Dict[str, float]:
    y_pred = (y_score >= threshold).astype(int)
    return metric_from_predictions(y_true, y_pred, y_score, prefix)


def metric_from_predictions(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray, prefix: str) -> Dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else float("nan")
    try:
        roc_auc = roc_auc_score(y_true, y_score)
    except ValueError:
        roc_auc = float("nan")
    try:
        pr_auc = average_precision_score(y_true, y_score)
    except ValueError:
        pr_auc = float("nan")
    return {
        f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}_precision": precision_score(y_true, y_pred, zero_division=0),
        f"{prefix}_recall": recall_score(y_true, y_pred, zero_division=0),
        f"{prefix}_f1": f1_score(y_true, y_pred, zero_division=0),
        f"{prefix}_roc_auc": roc_auc,
        f"{prefix}_pr_auc": pr_auc,
        f"{prefix}_specificity": specificity,
        f"{prefix}_tn": int(tn),
        f"{prefix}_fp": int(fp),
        f"{prefix}_fn": int(fn),
        f"{prefix}_tp": int(tp),
    }


def condition_thresholds_f1(y_true: np.ndarray, y_score: np.ndarray, condition_labels: Sequence[int]):
    global_threshold, _ = best_threshold_f1(y_true, y_score)
    condition_labels = np.asarray(condition_labels, dtype=int)
    thresholds = {}
    for cluster_id in range(N_CONDITIONS):
        mask = condition_labels == cluster_id
        if mask.sum() == 0 or len(np.unique(y_true[mask])) < 2:
            thresholds[cluster_id] = global_threshold
        else:
            thresholds[cluster_id], _ = best_threshold_f1(y_true[mask], y_score[mask])
    return thresholds, global_threshold


def predict_with_condition_thresholds(y_score: np.ndarray, condition_labels: Sequence[int], thresholds: Dict[int, float], default_threshold: float) -> np.ndarray:
    condition_labels = np.asarray(condition_labels, dtype=int)
    applied = np.array([thresholds.get(int(label), default_threshold) for label in condition_labels], dtype=np.float32)
    return (y_score >= applied).astype(int)


def fit_xgboost(x_train, y_train, x_val, y_val):
    n_neg = int(np.sum(y_train == 0))
    n_pos = int(np.sum(y_train == 1))
    scale_pos_weight = max(1.0, n_neg / max(n_pos, 1))
    clf = XGBClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        early_stopping_rounds=40,
    )
    clf.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    return clf


def evaluate_feature_set(dataset, feature_set, x_train, x_val, x_test_all, labels, condition_labels):
    y_train, y_val, y_test_all = labels
    cond_val, cond_test_all = condition_labels
    clf = fit_xgboost(x_train, y_train, x_val, y_val)
    val_score = clf.predict_proba(x_val)[:, 1]
    global_th, global_val_f1 = best_threshold_f1(y_val, val_score)
    all_score = clf.predict_proba(x_test_all)[:, 1]
    if CONDITION_AWARE_THRESHOLDS:
        condition_thresholds, default_th = condition_thresholds_f1(y_val, val_score, cond_val)
        val_pred = predict_with_condition_thresholds(val_score, cond_val, condition_thresholds, default_th)
        all_pred = predict_with_condition_thresholds(all_score, cond_test_all, condition_thresholds, default_th)
        val_f1 = f1_score(y_val, val_pred, zero_division=0)
        threshold_mode = "condition-aware"
    else:
        condition_thresholds = {i: global_th for i in range(N_CONDITIONS)}
        all_pred = (all_score >= global_th).astype(int)
        val_f1 = global_val_f1
        threshold_mode = "global"
    row = {
        "Dataset": dataset,
        "Feature Set": feature_set,
        "RUL Threshold": RUL_THRESHOLD,
        "Threshold Mode": threshold_mode,
        "Global Validation Threshold": global_th,
        "Condition Thresholds": json.dumps({str(k): round(float(v), 5) for k, v in condition_thresholds.items()}),
        "Validation F1": val_f1,
        "Train Positives": int(np.sum(y_train == 1)),
        "Train Negatives": int(np.sum(y_train == 0)),
        "Feature Dim": int(x_train.shape[1]),
    }
    row.update(metric_from_predictions(y_test_all, all_pred, all_score, "all"))
    return row


## 5. Run FD001-FD004 Ablation Experiments

This is the main experiment cell. It may take several minutes because it embeds every split and trains three XGBoost models per dataset.

**Where the FD001-FD004 results appear:** after this cell finishes, the next cell named **Quick Results View** displays the per-dataset results over the full test trajectories. The export cell also writes the same results to `article_assets/semantic_trend_xgboost/tables/semantic_trend_ablation_results.csv`, `.md`, and `.tex`.

In [ ]:
def prepare_dataset(dataset: str):
    train_df, test_df, op_cols, all_sensor_cols = load_cmapss(dataset, DATA_DIR)
    train_units, val_units = split_train_validation(train_df)
    train_part = train_df[train_df["unit_number"].isin(train_units)].copy()
    val_part = train_df[train_df["unit_number"].isin(val_units)].copy()
    sensor_cols = select_sensor_columns(train_part, all_sensor_cols)
    train_norm, val_norm, test_norm = maybe_normalize_conditions(dataset, train_part, val_part, test_df, op_cols, sensor_cols)
    cond_train = train_norm["_condition_cluster"].astype(int).values
    cond_val = val_norm["_condition_cluster"].astype(int).values
    cond_test_all = test_norm["_condition_cluster"].astype(int).values
    print(f"[{dataset}] sensors={len(sensor_cols)} train={len(train_norm):,} val={len(val_norm):,} test_all={len(test_norm):,}")
    print(f"[{dataset}] multi-window numeric features: {WINDOW_SIZES}; semantic text window: {TEXT_WINDOW_SIZE}")
    win_train = extract_multi_window_features(train_norm, sensor_cols)
    win_val = extract_multi_window_features(val_norm, sensor_cols)
    win_test_all = extract_multi_window_features(test_norm, sensor_cols)
    text_win_train = extract_window_features(train_norm, sensor_cols, TEXT_WINDOW_SIZE)
    text_win_val = extract_window_features(val_norm, sensor_cols, TEXT_WINDOW_SIZE)
    text_win_test_all = extract_window_features(test_norm, sensor_cols, TEXT_WINDOW_SIZE)

    return {
        "dataset": dataset,
        "sensor_cols": sensor_cols,
        "train_units": train_units,
        "val_units": val_units,
        "win_train": win_train,
        "win_val": win_val,
        "win_test_all": win_test_all,
        "condition_train": cond_train,
        "condition_val": cond_val,
        "condition_test_all": cond_test_all,
        "texts_train": build_texts(text_win_train, sensor_cols, cond_train),
        "texts_val": build_texts(text_win_val, sensor_cols, cond_val),
        "texts_test_all": build_texts(text_win_test_all, sensor_cols, cond_test_all),
        "rul_train": train_norm["RUL"].astype(float).values,
        "rul_val": val_norm["RUL"].astype(float).values,
        "rul_test_all": test_norm["RUL"].astype(float).values,
    }


def scaled_numeric_features(pkg):
    scaler = StandardScaler()
    x_train_window = scaler.fit_transform(pkg["win_train"].values.astype(np.float32))
    x_val_window = scaler.transform(pkg["win_val"].values.astype(np.float32))
    x_test_window = scaler.transform(pkg["win_test_all"].values.astype(np.float32))
    cond_train = condition_one_hot(pkg["condition_train"]).values
    cond_val = condition_one_hot(pkg["condition_val"]).values
    cond_test_all = condition_one_hot(pkg["condition_test_all"]).values
    x_train = np.concatenate([x_train_window, cond_train], axis=1)
    x_val = np.concatenate([x_val_window, cond_val], axis=1)
    x_test_all = np.concatenate([x_test_window, cond_test_all], axis=1)
    return x_train, x_val, x_test_all


def run_dataset_ablation(pkg, embedder):
    dataset = pkg["dataset"]
    labels = (
        labels_from_rul(pkg["rul_train"]),
        labels_from_rul(pkg["rul_val"]),
        labels_from_rul(pkg["rul_test_all"]),
    )
    x_num_train, x_num_val, x_num_all = scaled_numeric_features(pkg)
    feature_sets = {"numeric": (x_num_train, x_num_val, x_num_all)}

    if any(m in METHODS for m in ["semantic", "hybrid"]):
        prefix = f"mw{'-'.join(map(str, WINDOW_SIZES))}_tw{TEXT_WINDOW_SIZE}_cond_top{TOP_K_TEXT_SENSORS}_{HF_MODEL.split('/')[-1]}_{dataset}"
        emb_train = embedder.encode(pkg["texts_train"], f"{prefix}_train")
        emb_val = embedder.encode(pkg["texts_val"], f"{prefix}_val")
        emb_all = embedder.encode(pkg["texts_test_all"], f"{prefix}_test_all")
        feature_sets["semantic"] = (emb_train, emb_val, emb_all)
        feature_sets["hybrid"] = (
            np.concatenate([emb_train, x_num_train], axis=1),
            np.concatenate([emb_val, x_num_val], axis=1),
            np.concatenate([emb_all, x_num_all], axis=1),
        )

    rows = []
    for method in METHODS:
        print(f"[{dataset}] training {method}")
        condition_labels = (pkg["condition_val"], pkg["condition_test_all"])
        rows.append(evaluate_feature_set(dataset, method, *feature_sets[method], labels, condition_labels))
    return rows


start_time = time.time()
embedder = TextEmbedder(HF_MODEL, OUT_DIR / "cache") if any(m in METHODS for m in ["semantic", "hybrid"]) else None

all_rows = []
for dataset in DATASETS:
    print("\n===", dataset, "===")
    package = prepare_dataset(dataset)
    all_rows.extend(run_dataset_ablation(package, embedder))

results_df = pd.DataFrame(all_rows)
runtime_minutes = (time.time() - start_time) / 60.0
print(f"Runtime: {runtime_minutes:.1f} minutes")
results_df[["Dataset", "Feature Set", "all_f1", "all_precision", "all_recall", "all_roc_auc", "all_pr_auc", "Feature Dim"]]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== FD001 ===
[FD001] sensors=15 train=16,561 val=4,070 test_all=13,096
[FD001] multi-window numeric features: [10, 30, 60]; semantic text window: 30
[FD001] training numeric
[FD001] training semantic
[FD001] training hybrid

=== FD002 ===


/tmp/ipykernel_28228/4083728821.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.76383951 -1.47678703 -0.76383951 ...  2.08795057  2.08795057
  2.08795057]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[mask, list(sensor_cols)] = scaler.transform(out.loc[mask, list(sensor_cols)].fillna(0.0))
/tmp/ipykernel_28228/4083728821.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.18973455 -0.76383951 -1.47678703 ...  2.08795057  2.08795057
  1.37500305]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[mask, list(sensor_cols)] = scaler.transform(out.loc[mask, list(sensor_cols)].fillna(0.0))
/tmp/ipykernel_28228/4083728821.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error 

[FD002] sensors=21 train=43,464 val=10,295 test_all=33,991
[FD002] multi-window numeric features: [10, 30, 60]; semantic text window: 30
[FD002] training numeric
[FD002] training semantic
[FD002] training hybrid

=== FD003 ===
[FD003] sensors=16 train=20,012 val=4,708 test_all=16,596
[FD003] multi-window numeric features: [10, 30, 60]; semantic text window: 30
[FD003] training numeric
[FD003] training semantic
[FD003] training hybrid

=== FD004 ===


/tmp/ipykernel_28228/4083728821.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.27190928 -1.04409768 -1.65800348 ...  1.41152551  1.41152551
  2.0254313 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[mask, list(sensor_cols)] = scaler.transform(out.loc[mask, list(sensor_cols)].fillna(0.0))
/tmp/ipykernel_28228/4083728821.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.79761971 -0.43019189  0.18371391 ... -0.43019189  0.79761971
  0.79761971]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[mask, list(sensor_cols)] = scaler.transform(out.loc[mask, list(sensor_cols)].fillna(0.0))
/tmp/ipykernel_28228/4083728821.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error 

[FD004] sensors=21 train=49,294 val=11,955 test_all=41,214
[FD004] multi-window numeric features: [10, 30, 60]; semantic text window: 30
[FD004] training numeric
[FD004] training semantic
[FD004] training hybrid
Runtime: 1.5 minutes


,Dataset,Feature Set,all_f1,all_precision,all_recall,all_roc_auc,all_pr_auc,Feature Dim
0,FD001,numeric,0.823899,0.861842,0.789157,0.997912,0.929221,231
1,FD001,semantic,0.480186,0.391635,0.620482,0.965312,0.540913,384
2,FD001,hybrid,0.829721,0.853503,0.807229,0.998167,0.936681,615
3,FD002,numeric,0.840838,0.815744,0.867525,0.997537,0.940354,321
4,FD002,semantic,0.601516,0.554779,0.656854,0.971743,0.666278,384
5,FD002,hybrid,0.843384,0.825551,0.862006,0.997454,0.938863,705
6,FD003,numeric,0.861063,0.859589,0.862543,0.999169,0.955218,246
7,FD003,semantic,0.570667,0.466231,0.735395,0.982286,0.645938,384
8,FD003,hybrid,0.867395,0.794286,0.955326,0.999211,0.957952,630
9,FD004,numeric,0.768605,0.772196,0.765046,0.996000,0.825091,321


## Quick Results View

Run this after the experiment cell. It shows FD001-FD004 results over the full test trajectories directly inside the notebook.

In [ ]:
result_cols = [
    "Dataset",
    "Feature Set",
    "Threshold Mode",
    "all_f1",
    "all_precision",
    "all_recall",
    "all_roc_auc",
    "all_pr_auc",
    "all_specificity",
    "Feature Dim",
]
visible_results = results_df[result_cols].copy()
for col in visible_results.select_dtypes(include=["float", "float64", "float32"]).columns:
    visible_results[col] = visible_results[col].round(4)
display(visible_results.sort_values(["Dataset", "Feature Set"]).reset_index(drop=True))

all_f1_table = results_df.pivot(index="Dataset", columns="Feature Set", values="all_f1")
all_f1_table = all_f1_table[[c for c in ["numeric", "semantic", "hybrid"] if c in all_f1_table.columns]].round(4)
display(all_f1_table)


,Dataset,Feature Set,Threshold Mode,all_f1,all_precision,all_recall,all_roc_auc,all_pr_auc,all_specificity,Feature Dim
0,FD001,hybrid,condition-aware,0.8297,0.8535,0.8072,0.9982,0.9367,0.9964,615
1,FD001,numeric,condition-aware,0.8239,0.8618,0.7892,0.9979,0.9292,0.9967,231
2,FD001,semantic,condition-aware,0.4802,0.3916,0.6205,0.9653,0.5409,0.9749,384
3,FD002,hybrid,condition-aware,0.8434,0.8256,0.8620,0.9975,0.9389,0.9940,705
4,FD002,numeric,condition-aware,0.8408,0.8157,0.8675,0.9975,0.9404,0.9935,321
5,FD002,semantic,condition-aware,0.6015,0.5548,0.6569,0.9717,0.6663,0.9826,384
6,FD003,hybrid,condition-aware,0.8674,0.7943,0.9553,0.9992,0.9580,0.9956,630
7,FD003,numeric,condition-aware,0.8611,0.8596,0.8625,0.9992,0.9552,0.9975,246
8,FD003,semantic,condition-aware,0.5707,0.4662,0.7354,0.9823,0.6459,0.9850,384
9,FD004,hybrid,condition-aware,0.7736,0.7800,0.7674,0.9959,0.8245,0.9954,705


Feature Set,numeric,semantic,hybrid
Dataset,,,
FD001,0.8239,0.4802,0.8297
FD002,0.8408,0.6015,0.8434
FD003,0.8611,0.5707,0.8674
FD004,0.7686,0.5982,0.7736


## 6. Export Article Tables, Figures, and Draft Text

In [ ]:
def format_float(value):
    if isinstance(value, (float, np.floating)):
        if math.isnan(float(value)):
            return ""
        return round(float(value), 4)
    return value


def clean_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].map(format_float)
    return out


def markdown_table(df: pd.DataFrame) -> str:
    df = clean_table(df)
    headers = list(df.columns)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(row[h]) for h in headers) + " |")
    return "\n".join(lines)


def write_table(df: pd.DataFrame, base_path: Path) -> None:
    base_path.parent.mkdir(parents=True, exist_ok=True)
    clean = clean_table(df)
    clean.to_csv(base_path.with_suffix(".csv"), index=False)
    base_path.with_suffix(".md").write_text(markdown_table(clean) + "\n", encoding="utf-8")
    clean.to_latex(base_path.with_suffix(".tex"), index=False, escape=True)


summary_df = (
    results_df.groupby("Feature Set", as_index=False)
    .agg(
        Mean_All_F1=("all_f1", "mean"),
        Mean_All_ROC_AUC=("all_roc_auc", "mean"),
        Mean_All_PR_AUC=("all_pr_auc", "mean"),
    )
    .sort_values("Mean_All_F1", ascending=False)
)
f1_pivot_df = results_df.pivot(index="Dataset", columns="Feature Set", values="all_f1").reset_index()

tables_dir = OUT_DIR / "tables"
figures_dir = OUT_DIR / "figures"
write_table(results_df, tables_dir / "semantic_trend_ablation_results")
write_table(summary_df, tables_dir / "semantic_trend_ablation_summary")
write_table(f1_pivot_df, tables_dir / "semantic_trend_all_f1_pivot")

hyperparams_df = pd.DataFrame(
    [
        ("Task", "Supervised near-failure classification"),
        ("Near-failure label", f"RUL <= {RUL_THRESHOLD} cycles"),
        ("Numeric window sizes", ", ".join(map(str, WINDOW_SIZES))),
        ("Semantic text window", TEXT_WINDOW_SIZE),
        ("Validation split", f"{VAL_ENGINE_RATIO:.0%} engines from training trajectories"),
        ("Operating-condition normalization", "KMeans + per-condition StandardScaler, fit on train split only"),
        ("Condition clusters", N_CONDITIONS),
        ("Condition features", "KMeans cluster one-hot features appended to numeric vectors"),
        ("Window statistics", "mean, std, slope, max, min per selected sensor and window size"),
        ("Semantic encoder", HF_MODEL),
        ("Semantic text", f"Operating-condition cluster plus top {TOP_K_TEXT_SENSORS} absolute-slope sensor trends"),
        ("Classifier", "XGBoost binary logistic"),
        ("XGBoost estimators", N_ESTIMATORS),
        ("Probability threshold", "Per-condition validation F1 threshold" if CONDITION_AWARE_THRESHOLDS else "Global validation F1 threshold"),
        ("Random seed", SEED),
    ],
    columns=["Item", "Value"],
)
write_table(hyperparams_df, tables_dir / "semantic_trend_hyperparameters")

metadata = {
    "datasets": DATASETS,
    "methods": METHODS,
    "rul_threshold": RUL_THRESHOLD,
    "window_sizes": WINDOW_SIZES,
    "text_window_size": TEXT_WINDOW_SIZE,
    "condition_aware_thresholds": CONDITION_AWARE_THRESHOLDS,
    "val_engine_ratio": VAL_ENGINE_RATIO,
    "top_k_text_sensors": TOP_K_TEXT_SENSORS,
    "semantic_encoder": HF_MODEL,
    "seed": SEED,
    "python": platform.python_version(),
    "runtime_minutes": runtime_minutes,
}
(OUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

if plt is not None:
    figures_dir.mkdir(parents=True, exist_ok=True)
    pivot = results_df.pivot(index="Dataset", columns="Feature Set", values="all_f1")
    pivot = pivot[[c for c in ["numeric", "semantic", "hybrid"] if c in pivot.columns]]
    ax = pivot.plot(kind="bar", figsize=(8, 4), width=0.78)
    ax.set_ylabel("All-cycle F1-score")
    ax.set_ylim(0.0, 1.05)
    ax.set_xlabel("")
    ax.legend(title="")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(figures_dir / "fig_semantic_trend_ablation_f1.svg")
    plt.close()

    hybrid = results_df[results_df["Feature Set"] == "hybrid"]
    if not hybrid.empty:
        fig, axes = plt.subplots(1, len(hybrid), figsize=(3.1 * len(hybrid), 3.1))
        axes = np.atleast_1d(axes)
        for ax, (_, row) in zip(axes, hybrid.iterrows()):
            cm = np.array([[row["all_tn"], row["all_fp"]], [row["all_fn"], row["all_tp"]]], dtype=float)
            ax.imshow(cm, cmap="Blues")
            ax.set_title(row["Dataset"])
            ax.set_xticks([0, 1], labels=["Normal", "Near-failure"], rotation=30, ha="right")
            ax.set_yticks([0, 1], labels=["Normal", "Near-failure"])
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, str(int(cm[i, j])), ha="center", va="center")
        plt.tight_layout()
        plt.savefig(figures_dir / "fig_semantic_trend_hybrid_all_confusion.svg")
        plt.close()

best_method = summary_df.iloc[0]["Feature Set"]
best_f1 = summary_df.iloc[0]["Mean_All_F1"]
report = f"""# Semantic Trend XGBoost Article Result Report

## Scope

This report summarizes the article-ready CMAPSS near-failure detection experiment for the semantic trend embedding + XGBoost pipeline. The task is supervised binary classification: a cycle is labelled near-failure when RUL <= {RUL_THRESHOLD} cycles.

## Method

The pipeline computes rolling statistics at window sizes {WINDOW_SIZES} for each selected sensor. For FD002 and FD004, operating-condition normalization is fit only on the engine-level training split and then applied to validation and test data. The inferred condition cluster is appended as one-hot numeric features. Semantic trend sentences include the operating-condition cluster and the top {TOP_K_TEXT_SENSORS} absolute-slope sensors from a {TEXT_WINDOW_SIZE}-cycle window, then are embedded with `{HF_MODEL}`. XGBoost is evaluated with numeric-only, semantic-only, and hybrid feature sets. Probability thresholds are selected per condition cluster on the validation split.

## Main Result

The strongest mean all-cycle F1-score is obtained by `{best_method}` with mean F1 = {best_f1:.4f}. The full ablation table is available at `tables/semantic_trend_ablation_results.tex`, and the compact F1 comparison is available at `tables/semantic_trend_all_f1_pivot.tex`.

## Article Claim

The defensible claim is that language-model sentence embeddings can be used as a textual trend representation for CMAPSS near-failure detection, and their value is assessed through numeric-only, semantic-only, and hybrid ablation. Avoid calling the method a generative LLM system; the encoder is a sentence embedding model.
"""
(OUT_DIR / "semantic_trend_results_report.md").write_text(report, encoding="utf-8")

draft = f"""# Semantic Trend Embedding and XGBoost for Turbofan Near-Failure Detection

## Abstract

This study proposes a semantic trend embedding assisted XGBoost framework for near-failure detection in aircraft turbofan engines using the NASA C-MAPSS dataset. A near-failure state is defined as RUL <= {RUL_THRESHOLD} cycles. For each engine cycle, the method extracts multi-window rolling degradation statistics using windows {WINDOW_SIZES}, appends operating-condition cluster features, and converts the strongest {TEXT_WINDOW_SIZE}-cycle sensor trends into compact textual descriptions that also include the condition cluster. These descriptions are encoded using the `{HF_MODEL}` sentence embedding model and evaluated alone and in combination with numeric window statistics. Experiments on FD001-FD004 evaluate all available test cycles and show that the best feature set achieves a mean all-cycle F1-score of {best_f1:.4f}. Ablation results determine whether semantic trend embeddings add value beyond numeric and condition-aware degradation statistics.

## Contributions

1. A leakage-controlled CMAPSS near-failure detection protocol with engine-level train-validation splitting and train-only preprocessing.
2. A semantic trend representation that converts rolling sensor slopes and variability into sentence embeddings.
3. A cross-dataset ablation comparing numeric-only, semantic-only, and hybrid XGBoost classifiers on FD001-FD004.

## Wording Note

Do not describe the method as unsupervised anomaly detection. The correct task name is supervised near-failure classification. Do not claim a generative LLM is used; the method uses a pretrained sentence embedding model.
"""
(OUT_DIR / "draft_semantic_trend_article.md").write_text(draft, encoding="utf-8")

print("Wrote article assets to", OUT_DIR.resolve())
display(clean_table(summary_df))
display(clean_table(f1_pivot_df))


Wrote article assets to /content/article_assets/semantic_trend_xgboost


,Feature Set,Mean_All_F1,Mean_All_ROC_AUC,Mean_All_PR_AUC
0,hybrid,0.8285,0.9977,0.9145
1,numeric,0.8236,0.9977,0.9125
2,semantic,0.5626,0.9753,0.6202


Feature Set,Dataset,hybrid,numeric,semantic
0,FD001,0.8297,0.8239,0.4802
1,FD002,0.8434,0.8408,0.6015
2,FD003,0.8674,0.8611,0.5707
3,FD004,0.7736,0.7686,0.5982
